# R14 · Red neuronal con el mismo diseño

**Proyecto Final Integrador · Machine Learning I · Universidad EAN**
Integrantes: Claudia Naranjo, Yery Pedraza y Nicolás Ruiz.

Red de referencia para el score de pago tardío (B), con el diseño aprobado antes de ver resultados. Sus resultados entran en
[`informe_avance_M2B.md`](informe_avance_M2B.md).

| Sección | Criterio de R14 |
|---|---|
| §1 Diseño | Misma partición, columnas, métrica; escala y codificación en el `Pipeline` |
| §2 Sensibilidad | Tabla de sensibilidad a épocas, capacidad y regularización |
| §3 Red elegida con 5 semillas | Media y desviación con al menos tres semillas |
| §4 Modelo final del módulo | ¿La red mejora a la logística por más que la variación? |
| §5 Auditoría | Exhaustividad por grupo a un cupo declarado; monto perdido por empresa |

**La prueba no se usa.** **Confidencialidad:** solo cifras agregadas.

## 0. Configuración

In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datos_crudos").is_dir())
sys.path.insert(0, str(RAIZ / "src"))

from cartera import datos
from cartera import diseno as dis
from cartera.config import SEMILLA
from cartera.diseno import CAT, NUM

np.random.seed(SEMILLA)
SEMILLAS = [SEMILLA + i for i in range(5)]   # 2026 a 2030

print("Semilla", SEMILLA, "| semillas de la red", SEMILLAS, "| Python", sys.version.split()[0],
      "| pandas", pd.__version__, "| numpy", np.__version__, "| scikit-learn", sklearn.__version__)

Semilla 2026 | semillas de la red [2026, 2027, 2028, 2029, 2030] | Python 3.14.7 | pandas 3.0.6 | numpy 2.5.3 | scikit-learn 1.9.1


In [2]:
saldos, movs = datos.leer_todo()
conj = dis.conjuntos(saldos, movs)
subent, val, ent = conj["subentrenamiento"], conj["validación"], conj["entrenamiento completo"]
X, y = ent[NUM + CAT], ent.tardia
cv = dis.pliegues(ent, n=5)   # los mismos 5 pliegues de R13
print("Entrenamiento completo:", len(ent), "facturas | predictoras:", len(NUM) + len(CAT))

Entrenamiento completo: 8930 facturas | predictoras: 11


## 1. Diseño de la red

- **Mismo diseño que R13:** las 11 predictoras, los 5 pliegues agrupados por
  trimestre y AUC-PR.
- **`Pipeline`:** imputación (mediana, con indicador de faltante), escala y codificación one-hot
  (`dis.prep_lineal`), ajustadas solo con el entrenamiento de cada pliegue. La red mide distancias
  y gradientes: sin escala, las variables grandes (saldo) dominarían.
- **Desbalance:** las tardías pesan más con pesos de muestra «balanceados», como `class_weight`
  en los demás candidatos.
- **Sin `early_stopping`:** la parada temprana de scikit-learn separa su validación al azar por fila
  y mezclaría trimestres; el número de épocas se elige con los pliegues (§2).
- **Optimizador:** Adam, con la tasa de aprendizaje por defecto (0,001).

In [3]:
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.utils.class_weight import compute_sample_weight

# Con pocas épocas Adam no «converge» y avisa; es esperado en la tabla de sensibilidad
warnings.filterwarnings("ignore", category=ConvergenceWarning)

def red(capas=(32,), alpha=1e-3, epocas=100, semilla=SEMILLA):
    est = MLPClassifier(hidden_layer_sizes=capas, alpha=alpha, max_iter=epocas,
                        early_stopping=False, random_state=semilla)
    return Pipeline([("prep", dis.prep_lineal()), ("est", est)])

pesos = compute_sample_weight("balanced", y)

def cv_ap(modelo, con_pesos=True):
    """AUC-PR en los 5 pliegues. La red recibe pesos de muestra; la logística ya usa class_weight."""
    params = {"est__sample_weight": pesos} if con_pesos else None
    r = cross_validate(modelo, X, y, cv=cv, scoring="average_precision", params=params)
    return r["test_score"], r["fit_time"]

## 2. Tabla de sensibilidad

Desde una configuración base (una capa de 32 neuronas, α = 0,001, 100 épocas), se cambia un factor
a la vez. Cada configuración se evalúa con 3 semillas (2026 a 2028) en los 5 pliegues. Se reporta la
AUC-PR media, la desviación entre semillas (de las medias por semilla) y entre pliegues (de las
medias por pliegue).

In [4]:
BASE = {"capas": (32,), "alpha": 1e-3, "epocas": 100}
variantes = [("base", "—", {})]
variantes += [("épocas", e, {"epocas": e}) for e in [25, 50, 200]]
variantes += [("capacidad", c, {"capas": c}) for c in [(8,), (64, 32)]]
variantes += [("regularización α", a, {"alpha": a}) for a in [1e-4, 1e-2, 1.0]]

def evaluar_config(cambios, semillas):
    conf = {**BASE, **cambios}
    puntajes, tiempos = [], []
    for s in semillas:
        ap, t = cv_ap(red(**conf, semilla=s))
        puntajes.append(ap); tiempos.append(t.mean())
    m = np.array(puntajes)   # semillas × pliegues
    return conf, m, float(np.mean(tiempos))

filas, matrices = [], {}
for factor, valor, cambios in variantes:
    conf, m, t = evaluar_config(cambios, SEMILLAS[:3])
    clave = (factor, str(valor))
    matrices[clave] = (conf, m)
    filas.append({"factor": factor, "valor": str(valor), "capas": str(conf["capas"]), "α": conf["alpha"],
                  "épocas": conf["epocas"], "AUC-PR media": m.mean(),
                  "desv. entre semillas": m.mean(axis=1).std(), "desv. entre pliegues": m.mean(axis=0).std(),
                  "segundos por ajuste": t})
sensibilidad = pd.DataFrame(filas)
sensibilidad.round(4)

,factor,valor,capas,α,épocas,AUC-PR media,desv. entre semillas,desv. entre pliegues,segundos por ajuste
0,base,—,"(32,)",0.0010,100,0.7209,0.0015,0.0353,0.3669
1,épocas,25,"(32,)",0.0010,25,0.7254,0.0048,0.0325,0.0962
2,épocas,50,"(32,)",0.0010,50,0.7263,0.0037,0.0346,0.1855
3,épocas,200,"(32,)",0.0010,200,0.7152,0.0023,0.0354,0.7209
4,capacidad,"(8,)","(8,)",0.0010,100,0.7226,0.0017,0.0352,0.2733
5,capacidad,"(64, 32)","(64, 32)",0.0010,100,0.6955,0.0090,0.0492,0.6575
6,regularización α,0.0001,"(32,)",0.0001,100,0.7208,0.0013,0.0352,0.3646
7,regularización α,0.01,"(32,)",0.0100,100,0.7207,0.0026,0.0356,0.3644
8,regularización α,1.0,"(32,)",1.0000,100,0.7307,0.0018,0.0353,0.3365


In [5]:
base_ap = sensibilidad.loc[sensibilidad.factor == "base", "AUC-PR media"].iloc[0]
peso = {}
for factor in ["épocas", "capacidad", "regularización α"]:
    valores = pd.concat([sensibilidad.loc[sensibilidad.factor == factor, "AUC-PR media"], pd.Series([base_ap])])
    peso[factor] = valores.max() - valores.min()
pd.Series(peso).sort_values(ascending=False).round(4).to_frame("rango de la AUC-PR media (incluye la base)")

,rango de la AUC-PR media (incluye la base)
capacidad,0.0271
épocas,0.0111
regularización α,0.0100


## 3. Red elegida, con 5 semillas

Se elige la configuración de mejor AUC-PR media de la tabla (solo validación) y se repite con las 5
semillas (2026 a 2030) en los 5 pliegues.

In [6]:
i = sensibilidad["AUC-PR media"].idxmax()
fila = sensibilidad.loc[i]
CONF = matrices[(fila.factor, fila.valor)][0]
_, M, t_red = evaluar_config({k: CONF[k] for k in CONF}, SEMILLAS)
ap_log, t_log = cv_ap(Pipeline([("prep", dis.prep_lineal()),
                                ("est", LogisticRegression(class_weight="balanced", max_iter=2000))]),
                     con_pesos=False)
pd.Series({
    "Configuración elegida": f"capas {CONF['capas']}, α = {CONF['alpha']}, {CONF['epocas']} épocas",
    "AUC-PR media (5 semillas × 5 pliegues)": round(M.mean(), 4),
    "Desviación entre semillas": round(M.mean(axis=1).std(), 4),
    "Desviación entre pliegues": round(M.mean(axis=0).std(), 4),
    "Segundos por ajuste": round(t_red, 2),
    "Regresión logística: AUC-PR media (mismos pliegues)": round(ap_log.mean(), 4),
    "Regresión logística: desviación entre pliegues": round(ap_log.std(), 4),
}, dtype=object).to_frame("valor")

,valor
Configuración elegida,"capas (32,), α = 1.0, 100 épocas"
AUC-PR media (5 semillas × 5 pliegues),0.7295
Desviación entre semillas,0.0023
Desviación entre pliegues,0.0346
Segundos por ajuste,0.34
Regresión logística: AUC-PR media (mismos pliegues),0.7203
Regresión logística: desviación entre pliegues,0.0213


In [7]:
pd.DataFrame(M, index=[f"semilla {s}" for s in SEMILLAS], columns=[f"pliegue {k}" for k in range(1, 6)]).round(3)

,pliegue 1,pliegue 2,pliegue 3,pliegue 4,pliegue 5
semilla 2026,0.670,0.730,0.741,0.739,0.773
semilla 2027,0.670,0.730,0.741,0.750,0.774
semilla 2028,0.660,0.722,0.738,0.750,0.773
semilla 2029,0.672,0.729,0.739,0.739,0.766
semilla 2030,0.662,0.721,0.736,0.741,0.771


## 4. Modelo final preliminar del módulo

**Regla (fijada antes de ver resultados, como en R13):** la red reemplaza a la regresión logística solo si su mejora media
supera la variación entre semillas **y** la variación entre pliegues.

In [8]:
mejora = M.mean() - ap_log.mean()
var_semillas, var_pliegues = M.mean(axis=1).std(), M.mean(axis=0).std()
gana_red = mejora > max(var_semillas, var_pliegues)
FINAL = "Red neuronal" if gana_red else "Regresión logística"
pd.Series({
    "Mejora de la red sobre la logística": round(mejora, 4),
    "Variación entre semillas": round(var_semillas, 4),
    "Variación entre pliegues": round(var_pliegues, 4),
    "Pliegues en que la red (media de semillas) le gana a la logística": f"{int((M.mean(axis=0) > ap_log).sum())} de 5",
    "¿La mejora supera las dos variaciones?": bool(gana_red),
    "Modelo final preliminar del módulo": FINAL,
}, dtype=object).to_frame("valor")

,valor
Mejora de la red sobre la logística,0.0092
Variación entre semillas,0.0023
Variación entre pliegues,0.0346
Pliegues en que la red (media de semillas) le gana a la logística,4 de 5
¿La mejora supera las dos variaciones?,False
Modelo final preliminar del módulo,Regresión logística


## 5. Auditoría por grupo y monto perdido

La red elegida (semilla 2026) y la regresión logística se entrenan con el subentrenamiento de R12 y
se puntúan en su validación (2024-07 a 2025-06), como en R13 §6. **Cupo:** se marca el 30 % de las
facturas con mayor puntaje. **Exhaustividad** = parte de las tardías marcadas. El **monto** es el
valor inicial de la factura (montos escalados: son aproximados).

In [9]:
X_sub, y_sub = subent[NUM + CAT], subent.tardia
X_val, y_val = val[NUM + CAT], val.tardia
modelos = {
    "Red neuronal": red(**CONF, semilla=SEMILLA).fit(X_sub, y_sub, est__sample_weight=compute_sample_weight("balanced", y_sub)),
    "Regresión logística": Pipeline([("prep", dis.prep_lineal()),
                                     ("est", LogisticRegression(class_weight="balanced", max_iter=2000))]).fit(X_sub, y_sub),
}
grupos = {
    "Historia: 4 o menos pagadas": val.n_pagadas_hist <= 4,
    "Historia: 5 o más pagadas": val.n_pagadas_hist > 4,
    "Empresa A": val.empresa == "Empresa A",
    "Empresas B, C y D": val.empresa != "Empresa A",
    "Todas": pd.Series(True, index=val.index),
}
monto = np.expm1(val.log_monto_inicial)
filas = []
for nombre, modelo in modelos.items():
    p = modelo.predict_proba(X_val)[:, 1]
    marcada = pd.Series(p >= np.quantile(p, 0.70), index=val.index)
    for g, m in grupos.items():
        tard = m & (y_val == 1)
        filas.append({"modelo": nombre, "grupo": g, "tardías": int(tard.sum()),
                      "exhaustividad": round(marcada[tard].mean(), 3),
                      "falsos negativos": round(1 - marcada[tard].mean(), 3),
                      "monto tardío (millones)": round(monto[tard].sum() / 1e6, 1),
                      "monto tardío no marcado (millones)": round(monto[tard & ~marcada].sum() / 1e6, 1),
                      "% del monto tardío no marcado": round(monto[tard & ~marcada].sum() / monto[tard].sum() * 100, 1)})
auditoria = pd.DataFrame(filas).set_index(["modelo", "grupo"])
auditoria

tardías  exhaustividad  \
modelo              grupo                                                 
Red neuronal        Historia: 4 o menos pagadas       87          0.644   
                    Historia: 5 o más pagadas        734          0.669   
                    Empresa A                        764          0.686   
                    Empresas B, C y D                 57          0.404   
                    Todas                            821          0.666   
Regresión logística Historia: 4 o menos pagadas       87          0.759   
                    Historia: 5 o más pagadas        734          0.649   
                    Empresa A                        764          0.681   
                    Empresas B, C y D                 57          0.386   
                    Todas                            821          0.660   

                                                 falsos negativos  \
modelo              grupo                                           
Red neuronal        Historia: 4 o menos pagadas             0.356   
                    Historia: 5 o más pagadas               0.331   
                    Empresa A                               0.314   
                    Empresas B, C y D                       0.596   
                    Todas                                   0.334   
Regresión logística Historia: 4 o menos pagadas             0.241   
                    Historia: 5 o más pagadas               0.351   
                    Empresa A                               0.319   
                    Empresas B, C y D                       0.614   
                    Todas                                   0.340   

                                                 monto tardío (millones)  \
modelo              grupo                                                  
Red neuronal        Historia: 4 o menos pagadas                    665.8   
                    Historia: 5 o más pagadas                     3270.9   
                    Empresa A                                     3734.4   
                    Empresas B, C y D                              202.3   
                    Todas                                         3936.7   
Regresión logística Historia: 4 o menos pagadas                    665.8   
                    Historia: 5 o más pagadas                     3270.9   
                    Empresa A                                     3734.4   
                    Empresas B, C y D                              202.3   
                    Todas                                         3936.7   

                                                 monto tardío no marcado (millones)  \
modelo              grupo                                                             
Red neuronal        Historia: 4 o menos pagadas                               249.6   
                    Historia: 5 o más pagadas                                1011.0   
                    Empresa A                                                1150.2   
                    Empresas B, C y D                                         110.5   
                    Todas                                                    1260.7   
Regresión logística Historia: 4 o menos pagadas                               131.8   
                    Historia: 5 o más pagadas                                1040.8   
                    Empresa A                                                1054.1   
                    Empresas B, C y D                                         118.6   
                    Todas                                                    1172.7   

                                                 % del monto tardío no marcado  
modelo              grupo                                                       
Red neuronal        Historia: 4 o menos pagadas                           37.5  
                    Historia: 5 o más pagadas                             30.9  
                    Empresa A        